In [1]:
import tensorflow as tf # type: ignore
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from meridian.model import model
from meridian.analysis import visualizer, analyzer

In [2]:
home_dir = '/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer'
model_path = f'{home_dir}/model_objects/champ/'
# model_name = '0_0922_cpm_weighted_by_working_spend_national_False_Live_Nation_MasterAdvertiser.pkl'
model_name = '0_tigher_ec_cpm_weighted_by_working_spend_informative_national_False_Live_Nation_MasterAdvertiser.pkl'

In [3]:
# Load the model
mmm = model.load_mmm(f'{model_path}/{model_name}')


In [45]:
posterior = mmm.inference_data.posterior

pos_sample = posterior.sel(chain=0, draw=slice(0, 10))

# baseline
mu_t = tf.convert_to_tensor(pos_sample.mu_t, dtype=tf.float32)
tau_g = tf.convert_to_tensor(pos_sample.tau_g, dtype=tf.float32)
tau_gt = tf.expand_dims(tau_g, -1) + tf.expand_dims(mu_t, -2)

# media predictions
alpha_m = tf.constant(pos_sample.alpha_m, dtype=tf.float32)
ec_m = tf.constant(pos_sample.ec_m, dtype=tf.float32)
slope_m = tf.constant(pos_sample.slope_m, dtype=tf.float32)
beta_gm = tf.constant(pos_sample.beta_gm, dtype=tf.float32)

media_scaled = mmm.media_tensors.media_scaled
media_transformed = mmm.adstock_hill_media(
    media=media_scaled,
    alpha=alpha_m,
    ec=ec_m,
    slope=slope_m
)

media_contribution = tf.einsum('...gtm, ...gm -> ...gt', media_transformed, beta_gm)

pred_scaled = tau_gt + media_contribution
pred = mmm.kpi_transformer.inverse(pred_scaled)

In [48]:
tf.reduce_sum(pred, axis=(1, 2))

<tf.Tensor: shape=(11,), dtype=float32, numpy=
array([1.1154301e+09, 1.1398441e+09, 1.1340625e+09, 1.1358296e+09,
       1.1448233e+09, 1.1477267e+09, 1.1375698e+09, 1.1452960e+09,
       1.1359868e+09, 1.1393628e+09, 1.1452096e+09], dtype=float32)>

In [50]:
_analyzer = analyzer.Analyzer(mmm)
expected_outcome = _analyzer.expected_outcome()

expected_outcome[0, :11]


<tf.Tensor: shape=(11,), dtype=float32, numpy=
array([1.1154301e+09, 1.1398437e+09, 1.1340630e+09, 1.1358298e+09,
       1.1448247e+09, 1.1477256e+09, 1.1375706e+09, 1.1452970e+09,
       1.1359855e+09, 1.1393617e+09, 1.1452095e+09], dtype=float32)>

In [37]:
tau_g = tf.convert_to_tensor(pos_sample.tau_g, dtype=tf.float32)
tau_g[..., tf.newaxis].shape, tf.expand_dims(tau_g, -1).shape
np.allclose(tau_g[..., tf.newaxis], tf.expand_dims(tau_g, -1))

True